# CeNN Research Layers — Can a New Layer Match the Transformer?
**Focused experiments · SmolLM2-135M · Kimi Delta Attention / Gated DeltaNet-2**

Train **only three new proposed layers**, then measure the language model with one attention layer replaced. The pretrained Q/K/V/O projections, RoPE, MLPs and all other layers stay frozen.

| Candidate | New idea | Attention inside the replacement |
|:--|:--|:--|
| **CeNN–KDA** | Selective channel forgetting with a tied erase/write gate | Recurrent only |
| **CeNN–Delta2** | Independent erase and write gates | Recurrent only |
| **CeNN–Delta2 + Window** | Delta2 memory plus a learned local-window mixture | Hybrid: 32-token softmax window |

**Research hypothesis:** Delta2 + Window has the best chance of preserving pretrained quality; Delta2 is the primary candidate when the new layer must be softmax-free. These are hypotheses, not measured results.

**Start:** choose **Runtime → Change runtime type → GPU**, then run the cells in order. The default balanced profile is a screening run. No Hugging Face token is required for the public model and dataset.


## 1 · Prepare a fresh checkout
The checkout is isolated, so rerunning this cell does not reset another experiment. The exact source commit, model revision and dataset revision are recorded in the report. PyTorch already supplied by Colab is retained.


In [ ]:
import importlib, pathlib, subprocess, sys, tempfile

REPO_REF = "codex/cenn-research-layers-20260914"
WORK_PARENT = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
REPO_DIR = pathlib.Path(tempfile.mkdtemp(prefix="TinyCeNN-research-", dir=WORK_PARENT))
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_REF,
    "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(REPO_DIR), "transformers==4.57.6", "datasets>=3,<5",
    "pandas", "matplotlib", "pytest>=8"
], check=True)
for path in (REPO_DIR, REPO_DIR / "src"):
    sys.path.insert(0, str(path))
importlib.invalidate_caches()
SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Source commit:", SOURCE_COMMIT)
print("Checkout:", REPO_DIR)


## 2 · Choose the research budget
Keep **balanced** for the first run. **Smoke** verifies the pipeline and cannot establish parity. **Extended** increases training data, feature capacity and context.

Each candidate gets the same document order, seed and update budget. Changing `LAYERS` to `"0,18,29"` tests three independent replacement positions; it does not replace all three simultaneously.

To test only recurrent candidates, remove `cenn_delta2_window` from `VARIANTS`.


In [ ]:
import torch
from datetime import datetime, timezone

PROFILE = "balanced"  # "smoke", "balanced", or "extended"
LAYERS = "18"
VARIANTS = "cenn_kda,cenn_delta2,cenn_delta2_window"
SEED = 2026
ALLOW_CPU = False
SAVE_TO_DRIVE = False  # Set True to retain checkpoints throughout a long Colab run.

PROFILES = {
    "smoke": dict(context=64, test_contexts="64,128", feature_dims="32",
                  train_documents=8, validation_documents=4, test_documents=4,
                  steps=10, lm_steps=2, eval_every=5, window=16),
    "balanced": dict(context=256, test_contexts="256,512", feature_dims="64",
                     train_documents=64, validation_documents=12, test_documents=24,
                     steps=400, lm_steps=40, eval_every=50, window=32),
    "extended": dict(context=512, test_contexts="512,1024", feature_dims="64,128",
                     train_documents=256, validation_documents=32, test_documents=64,
                     steps=1200, lm_steps=120, eval_every=100, window=32),
}
CONFIG = PROFILES[PROFILE].copy()
assert torch.cuda.is_available() or ALLOW_CPU, "Select a GPU runtime, or set ALLOW_CPU=True for smoke only."
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("PyTorch:", torch.__version__)
print("Profile:", PROFILE, CONFIG)
print("Unique training input tokens:", CONFIG["train_documents"] * CONFIG["context"])
print("Transfer token presentations / candidate:", CONFIG["steps"] * CONFIG["context"])
print("Language-loss token presentations / candidate:", CONFIG["lm_steps"] * CONFIG["context"])
print("Candidate runs:", len(LAYERS.split(",")) * len(VARIANTS.split(",")) * len(CONFIG["feature_dims"].split(",")))
print("Runtime depends on the assigned GPU; the report records actual timing.")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_ROOT = pathlib.Path("/content/drive/MyDrive/TinyCeNN-LM/research-layers")
else:
    RESULT_ROOT = WORK_PARENT / "TinyCeNN-research-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)


## 3 · Check the mathematics before training
These CPU checks compare chunked updates and gradients with a tokenwise oracle, test causality and bounded state, and exercise both training phases inside a small randomly initialized Llama. They do not download a model and are not quality results.


In [ ]:
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    str(REPO_DIR / "tests/test_research_layers.py"),
    str(REPO_DIR / "tests/test_research_layer_benchmark.py"),
], cwd=REPO_DIR, check=True)


## 4 · Train the new layers and test their replacements
The workflow resolves exact model/data revisions, hashes documents into disjoint adaptation splits, trains new-layer parameters, selects checkpoints on validation NLL, and locks selection **before test evaluation**.

The final table measures real next-token NLL/perplexity after replacing one attention layer. Delta-rule memory can use signed coefficients, so attention-distribution KL from the previous positive-kernel notebook is not a valid comparison here.

**Longer context:** each test document is evaluated at both configured lengths. “Held out” means held out from this adaptation run; it does not guarantee absence from the original model's pretraining data.

Checkpoints and progress are saved as candidates finish. A new result directory is used for each execution.


In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = RESULT_ROOT / f"{PROFILE}-{RUN_ID}"
command = [
    sys.executable, str(REPO_DIR / "scripts/benchmark_cenn_research_layers.py"),
    "--layers", LAYERS, "--variants", VARIANTS, "--seed", str(SEED),
    "--output-dir", str(OUTPUT_DIR),
]
for key, value in CONFIG.items():
    command.extend(["--" + key.replace("_", "-"), str(value)])
print("Results:", OUTPUT_DIR)
print("Command:", subprocess.list2cmdline(command))
subprocess.run(command, cwd=REPO_DIR, check=True)


## 5 · See the decision and the exact metrics

| Measure | Better direction | Interpretation |
|:--|:--|:--|
| **PPL ratio** | Lower | 1.00 equals the original Transformer on this test |
| **ΔNLL with 95% interval** | Lower | Paired difference over test documents |
| **Output cosine / NMSE** | Higher / lower | How closely the new attention output matches |
| **Prefill / decode speedup** | Higher | A value below 1 means the implemented kernel is slower |
| **State vs FP16 KV** | Lower | Actual FP32 recurrent state against an FP16 Transformer cache |

The declared practical margin is **±0.02 nats** (about ±2% PPL). The full interval must lie inside that margin for the corresponding label. Fewer than eight test documents always gives “insufficient test documents.”

These intervals do not measure variability across training seeds. A completed experiment does not mean a candidate achieved parity.


In [ ]:
import json
import pandas as pd
from IPython.display import display

report = json.loads((OUTPUT_DIR / "research_layer_report.json").read_text())
results = pd.read_csv(OUTPUT_DIR / "research_layer_summary.csv")
validation = pd.read_csv(OUTPUT_DIR / "validation_summary.csv")
history = pd.read_csv(OUTPUT_DIR / "training_history.csv")
candidates = results[results["variant"] != "transformer_original"].copy()
selected = candidates[candidates["selected_on_validation"].eq(True)]

print("Validation-selected candidates:", report["validation_winners"])
print("Scope:", report["scope"])
display(selected[[
    "candidate", "context", "test_perplexity", "transformer_perplexity", "ppl_ratio",
    "delta_nll", "delta_nll_ci_low", "delta_nll_ci_high", "quality"
]].round(5))

print("All preregistered candidates — selected_on_validation identifies the decision made before testing:")
display(candidates[[
    "layer", "variant", "feature_dim", "context", "selected_on_validation",
    "trainable_parameters", "test_perplexity", "ppl_ratio", "output_cosine",
    "output_nmse", "grad_mean_cosine", "prefill_speedup", "decode_speedup",
    "state_vs_transformer_fp16", "quality"
]].round(5))

print("Reproduction:")
display(pd.DataFrame([{
    key: report.get(key) for key in
    ["source_commit", "model_revision", "dataset_revision", "gpu", "torch",
     "transformers", "teacher_dtype", "kernel_dtype", "unique_train_tokens"]
}]))
print(report["interpretation"])


## 6 · Compare quality, speed and memory
The charts use separate axes for separate objectives. They do not compress quality and speed into a heuristic “winner” score.

Timing is a warmed-up, synchronized FP32 kernel microbenchmark on one fixed document. It excludes Q/K/V/O projections and other model layers. It is not end-to-end generation throughput.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12,
})
COLORS = {"cenn_kda": "#276FBF", "cenn_delta2": "#008577", "cenn_delta2_window": "#D66A22"}
plot_files = []

for layer, data in candidates.groupby("layer"):
    data = data.sort_values(["context", "variant", "feature_dim"]).reset_index(drop=True)
    labels = [
        f"{row.variant.replace('cenn_', '')} · F{int(row.feature_dim)} · T{int(row.context)}"
        for _, row in data.iterrows()
    ]
    y = np.arange(len(data))
    colors = [COLORS[name] for name in data.variant]
    fig, axes = plt.subplots(2, 2, figsize=(15, max(8, len(data) * 0.8)))
    lo, hi = np.exp(data.delta_nll_ci_low.to_numpy()), np.exp(data.delta_nll_ci_high.to_numpy())
    ratio = data.ppl_ratio.to_numpy()
    errors = np.vstack([np.maximum(0, ratio - lo), np.maximum(0, hi - ratio)])
    axes[0, 0].barh(y, ratio, color=colors, xerr=errors, capsize=3, alpha=0.9)
    axes[0, 0].axvline(1.0, color="#263238", linestyle="--")
    axes[0, 0].axvspan(np.exp(-0.02), np.exp(0.02), color="#99BBAA", alpha=0.15)
    axes[0, 0].set(title="Held-out perplexity ratio · lower is better", xlabel="Candidate / Transformer")
    axes[0, 1].barh(y, data.output_cosine, color=colors)
    axes[0, 1].axvline(0.97, color="#263238", linestyle=":", label="0.97 fidelity reference")
    axes[0, 1].set(title="Attention-output fidelity · higher is better", xlabel="Output cosine")
    axes[0, 1].legend(fontsize=8)
    axes[1, 0].barh(y - 0.18, data.prefill_speedup, height=0.34, color="#276FBF", label="Prefill")
    axes[1, 0].barh(y + 0.18, data.decode_speedup, height=0.34, color="#D66A22", label="One-token decode")
    axes[1, 0].axvline(1.0, color="#263238", linestyle="--")
    axes[1, 0].set(title="Measured kernel speed · higher is better", xlabel="Transformer time / candidate time")
    axes[1, 0].legend(fontsize=8)
    axes[1, 1].barh(y, data.state_vs_transformer_fp16, color=colors)
    axes[1, 1].axvline(1.0, color="#263238", linestyle="--")
    axes[1, 1].set(title="Decode state · lower is better", xlabel="FP32 candidate state / FP16 Transformer KV")
    for ax in axes.flat:
        ax.set_yticks(y, labels)
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.16)
        ax.set_axisbelow(True)
    fig.suptitle(f"CeNN research candidates · attention layer {int(layer)}", fontsize=16, y=1.01)
    fig.tight_layout()
    for extension in ("png", "svg"):
        path = OUTPUT_DIR / f"comparison-layer-{int(layer)}.{extension}"
        fig.savefig(path, bbox_inches="tight")
        plot_files.append(path)
    plt.show()

transfer_history = history[history.phase == "transfer"]
if len(transfer_history):
    fig, ax = plt.subplots(figsize=(11, 4))
    for name, data in transfer_history.groupby("candidate"):
        ax.plot(data.step, data.output_nmse, marker=".", label=name)
    ax.set(xlabel="Transfer update", ylabel="Validation NMSE",
           title="Attention transfer: validation error, not training fit")
    ax.set_yscale("log")
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    path = OUTPUT_DIR / "transfer-learning-curves.png"
    fig.savefig(path, bbox_inches="tight")
    plot_files.append(path)
    plt.show()


## 7 · Optional: inspect your earlier CSV/JSON results
This imports existing reports for reference without retraining old variants. The old and new protocols differ, so their rows are displayed separately and never used to select the new layer.

Set `IMPORT_PREVIOUS_RESULTS=True` and upload your `learned_kernel_summary...` CSV or `learned_kernel_report...` JSON files when Colab asks.


In [ ]:
IMPORT_PREVIOUS_RESULTS = False
if IMPORT_PREVIOUS_RESULTS:
    from google.colab import files as colab_files
    import io
    uploaded = colab_files.upload()
    for name, content in uploaded.items():
        if name.lower().endswith(".csv"):
            old = pd.read_csv(io.BytesIO(content))
        elif name.lower().endswith(".json"):
            payload = json.loads(content)
            old = pd.DataFrame(payload.get("rows", payload.get("ranking", [])))
        else:
            continue
        print("Previous protocol:", name)
        columns = [key for key in [
            "layer", "variant", "effective_feature_dim", "output_cosine", "output_nmse",
            "attention_kl", "grad_mean_cosine", "runtime_ms"
        ] if key in old.columns]
        display(old[columns])


## 8 · Reload the selected new layer and download the results
Each checkpoint contains only the new layer's weights and constructor settings; the original model remains at the recorded Hugging Face revision. The ZIP includes the report, plots, token blocks and checkpoints.

If you selected Drive earlier, completed files also remain in the configured Drive folder. Intermediate checkpoints do not include optimizer state for exact training resumption.


In [ ]:
import shutil

selected_key = next(iter(report["validation_winners"].values()))
record = next(row for row in report["candidates"] if row["candidate"] == selected_key)
checkpoint = torch.load(OUTPUT_DIR / record["checkpoint"], map_location="cpu", weights_only=True)
from tinycenn_lm.research_layers import ResearchCeNNLayer
restored_core = ResearchCeNNLayer(**checkpoint["config"])
restored_core.load_state_dict(checkpoint["state_dict"])
print("Reloaded:", selected_key)
print("Core config:", checkpoint["config"])
print("Retained decode state bytes:", restored_core.recurrent_state_bytes())

ZIP_PATH = pathlib.Path(shutil.make_archive(
    str(OUTPUT_DIR.parent / (OUTPUT_DIR.name + "-results")), "zip", root_dir=OUTPUT_DIR
))
print("Result archive:", ZIP_PATH)
try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_PATH))
except ImportError:
    print("Download the ZIP from the notebook file browser.")


## Research notes and next decision

- If a recurrent candidate approaches the declared NLL margin, rerun the preregistered design with more training documents and additional seeds.
- If only the window variant succeeds, the evidence supports a hybrid replacement. It does not establish a completely softmax-free model.
- A single-layer result does not predict the result of replacing every layer. Error accumulation needs a separate joint experiment.
- Candidate selection uses validation data. After learning from test results, reserve a **new untouched test set** for the next architecture revision.
- A slower PyTorch reference with good quality can motivate a later optimized kernel; do not report paper throughput numbers as this implementation's performance.

These are CeNN-style memory-array adaptations. They are not a reproduction of a classical nearest-neighbor CeNN ODE or the cited papers' complete architectures.

**Primary sources:** [Gated DeltaNet-2, May 2026](https://arxiv.org/abs/2605.22791), [Kimi Linear / KDA, 2025](https://arxiv.org/abs/2510.26692), [LoLCATs, ICLR 2025](https://arxiv.org/abs/2410.10254), [Gated DeltaNet, ICLR 2025](https://arxiv.org/abs/2412.06464).

See [RESEARCH_LAYERS.md](https://github.com/vtavakkoli/TinyCeNN-LM/blob/codex/cenn-research-layers-20260914/RESEARCH_LAYERS.md) for equations, exact implementation choices, measurement limits, and command-line usage.
